# MVPA L2 Results Notebook

This notebook is now a read-only manuscript-facing review layer for the Hyak MVPA L2 workflow. It loads harmonized subject metrics, model statistics, diagnostics, and Markdown summaries produced by `code/hyak/*` and `code/scripts/run_mvpa_l2_posthyak.sh`.

The notebook must not fit models, run hypothesis tests, calculate corrected p-values, create permutation nulls, or export statistical tables. If a required statistics file is absent, rerun the Hyak/post-Hyak workflow rather than deriving the statistic here.


## Hyak Output Contract

Primary source scripts:

- `code/hyak/mvpa_L2_voxel_FearNetwork.py` produces the primary FearNetwork feature-space checkpoints and intermediates.
- `code/hyak/mvpa_L2_voxel_MemoryFearNetwork.py` produces the MemoryFearNetwork sensitivity checkpoints and intermediates.
- `code/hyak/mvpa_L2_voxel_WholeBrain_Schaefer.py` optionally produces the Schaefer/Tian whole-brain sensitivity checkpoints and intermediates.
- `code/scripts/run_mvpa_l2_posthyak.sh` harmonizes those outputs and writes the notebook-facing `harmonized/` and `stats/` artifacts.

Set `MVPA_L2_ROOT` to the completed post-Hyak result directory when needed.


In [ ]:
from pathlib import Path
import os
import warnings
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

pd.set_option('display.max_columns', 160)
pd.set_option('display.max_rows', 200)

cwd_candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((p for p in cwd_candidates if (p / 'PROJECT_CONTEXT.md').exists()), Path.cwd())

root_env = os.environ.get('MVPA_L2_ROOT')
mask_mode_env = os.environ.get('STAGE11_MASK_MODE') or os.environ.get('MVPA_L2_MASK_MODE')
default_roots = ['mvpa_l2_originalMask', 'mvpa_l2'] if mask_mode_env == 'original_notebook' else ['mvpa_l2', 'mvpa_l2_originalMask']

candidates = []
if root_env:
    candidates.append(Path(root_env).expanduser())
for name in default_roots:
    candidates.extend([
        PROJECT_ROOT / 'outputs' / name,
        PROJECT_ROOT / 'code' / 'outputs' / name,
        Path('/Users/xiaoqianxiao/projects/NARSAD/LSS/results') / name,
        Path('/gscratch/fang/NARSAD/MRI/derivatives/fMRI_analysis/LSS/results') / name,
        Path('/output_dir') / name,
    ])

def mvpa_root_ready(path):
    return (path / 'harmonized' / 'mvpa_l2_subject_metrics.csv').exists() or (path / 'stats').exists()

ready_candidates = [p for p in candidates if mvpa_root_ready(p)]
existing_candidates = [p for p in candidates if p.exists()]
MVPA_ROOT = ready_candidates[0] if ready_candidates else (existing_candidates[0] if existing_candidates else candidates[0])
HARMONIZED_DIR = MVPA_ROOT / 'harmonized'
STATS_DIR = MVPA_ROOT / 'stats'

if not MVPA_ROOT.exists():
    warnings.warn(f'MVPA_ROOT does not exist yet: {MVPA_ROOT}. Set MVPA_L2_ROOT to the completed post-Hyak output directory.')
elif not STATS_DIR.exists():
    warnings.warn(f'MVPA_ROOT exists but is missing stats/: {MVPA_ROOT}. Run code/scripts/run_mvpa_l2_posthyak.sh before reading results.')

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'MVPA_ROOT = {MVPA_ROOT}')
print(f'HARMONIZED_DIR = {HARMONIZED_DIR}')
print(f'STATS_DIR = {STATS_DIR}')


In [ ]:
ARTIFACT_CANDIDATES = {
    'subject_metrics': [HARMONIZED_DIR / 'mvpa_l2_subject_metrics.csv'],
    'scr_flags': [HARMONIZED_DIR / 'scr_sensitivity_groups.csv'],
    'aim1_primary': [STATS_DIR / 'aim1_primary_decoding.csv', STATS_DIR / 'aim1_decoding_primary.csv'],
    'aim1_feature_sensitivity': [STATS_DIR / 'aim1_sensitivity_feature_space.csv', STATS_DIR / 'aim1_mask_feature_sensitivity.csv'],
    'aim1_feature_sensitivity_wide': [STATS_DIR / 'aim1_sensitivity_feature_space_wide.csv', STATS_DIR / 'aim1_mask_feature_sensitivity_wide.csv'],
    'aim1_feature_sensitivity_raincloud': [STATS_DIR / 'aim1_sensitivity_feature_space_raincloud.csv', STATS_DIR / 'aim1_mask_feature_sensitivity_raincloud.csv'],
    'aim1_scr_sensitivity': [STATS_DIR / 'aim1_sensitivity_scr_cohort.csv', STATS_DIR / 'aim1_scr_sensitivity.csv'],
    'aim1_scr_sensitivity_raincloud': [STATS_DIR / 'aim1_sensitivity_scr_cohort_raincloud.csv', STATS_DIR / 'aim1_scr_sensitivity_raincloud.csv'],
    'aim1_primary_functional_drop_tests': [STATS_DIR / 'aim1_primary_decoding_functional_drop_tests.csv', STATS_DIR / 'aim1_decoding_primary_functional_drop_tests.csv'],
    'aim1_feature_functional_drop_tests': [STATS_DIR / 'aim1_sensitivity_feature_space_functional_drop_tests.csv', STATS_DIR / 'aim1_mask_feature_sensitivity_functional_drop_tests.csv'],
    'aim1_scr_functional_drop_tests': [STATS_DIR / 'aim1_sensitivity_scr_cohort_functional_drop_tests.csv', STATS_DIR / 'aim1_scr_sensitivity_functional_drop_tests.csv'],
    'aim1_feature_functional_drop_nulls': [STATS_DIR / 'aim1_sensitivity_feature_space_functional_drop_nulls.csv', STATS_DIR / 'aim1_mask_feature_sensitivity_functional_drop_nulls.csv'],
    'aim1_scr_functional_drop_nulls': [STATS_DIR / 'aim1_sensitivity_scr_cohort_functional_drop_nulls.csv', STATS_DIR / 'aim1_scr_sensitivity_functional_drop_nulls.csv'],
    'aim2_primary': [STATS_DIR / 'aim2_primary_group_difference.csv', STATS_DIR / 'aim2_group_difference.csv', STATS_DIR / 'Table2_Aim2_primary_statistics.csv'],
    'aim2_secondary_subject_metrics': [STATS_DIR / 'aim2_secondary_subject_metrics.csv'],
    'aim2_secondary': [STATS_DIR / 'aim2_secondary_group_difference.csv', STATS_DIR / 'Table_S2_secondary_support.csv', STATS_DIR / 'Table_aim2_secondary.csv'],
    'aim2_geometry_panel': [STATS_DIR / 'aim2_geometry_panel.csv'],
    'aim2_trajectory_panel': [STATS_DIR / 'aim2_trajectory_panel.csv', STATS_DIR / 'Table2_Aim2_trialwise_discrimination_statistics.csv'],
    'aim2_sensitivity': [STATS_DIR / 'aim2_sensitivity_group_difference.csv', STATS_DIR / 'Table_Aim2_Sensitivity_Stats.csv', STATS_DIR / 'TableS2_Aim2_Sensitivity_Stats.csv'],
    'aim2_sensitivity_primary': [STATS_DIR / 'aim2_sensitivity_primary_group_difference.csv'],
    'aim2_sensitivity_secondary': [STATS_DIR / 'aim2_sensitivity_secondary_group_difference.csv'],
    'aim2_haufe_scr_sensitivity': [STATS_DIR / 'aim2_haufe_scr_sensitivity.csv'],
    'aim2_haufe_scr_sensitivity_roi': [STATS_DIR / 'aim2_haufe_scr_sensitivity_roi_distribution.csv'],
    'neural_metric_registry': [STATS_DIR / 'neural_metric_registry.csv'],
    'aim3_primary': [STATS_DIR / 'aim3_primary_clinical_relevance.csv', STATS_DIR / 'aim3_clinical_relevance.csv', STATS_DIR / 'Table_Aim3_primary_statistics.csv'],
    'aim3_secondary': [STATS_DIR / 'aim3_secondary_clinical_relevance.csv', STATS_DIR / 'Table_Aim3_secondary_statistics.csv'],
    'aim3_sensitivity': [STATS_DIR / 'aim3_sensitivity_clinical_relevance.csv', STATS_DIR / 'Table_Aim3_sensitivity_statistics.csv'],
    'aim4_primary': [STATS_DIR / 'aim4_primary_scr_convergence.csv', STATS_DIR / 'aim4_scr_convergence.csv', STATS_DIR / 'Table4_Aim4_primary_neural_SCR_convergence.csv'],
    'aim4_secondary': [STATS_DIR / 'aim4_secondary_scr_convergence.csv', STATS_DIR / 'TableS4_Aim4_secondary_neural_SCR_convergence.csv'],
    'aim4_sensitivity': [STATS_DIR / 'aim4_sensitivity_scr_convergence.csv', STATS_DIR / 'TableS5_Aim4_sensitivity_neural_SCR_convergence.csv'],
    'aim4_convergence_matrix': [STATS_DIR / 'aim4_convergence_matrix.csv'],
    'aim4_convergence_matrix_wide': [STATS_DIR / 'aim4_convergence_matrix_wide.csv'],
    'aim4_convergence_md': [STATS_DIR / 'aim4_convergence_matrix.md'],
    'aim5_primary': [STATS_DIR / 'aim5_primary_oxytocin_modulation.csv', STATS_DIR / 'aim5_oxytocin_modulation.csv', STATS_DIR / 'Table_Aim5_primary_oxytocin_modulation.csv'],
    'aim5_secondary': [STATS_DIR / 'aim5_secondary_oxytocin_modulation.csv', STATS_DIR / 'Table_Aim5_secondary_oxytocin_modulation.csv'],
    'aim5_sensitivity': [STATS_DIR / 'aim5_sensitivity_oxytocin_modulation.csv', STATS_DIR / 'Table_Aim5_sensitivity_oxytocin_modulation.csv'],
    'aims_primary_all': [STATS_DIR / 'aims_primary_models_all.csv', STATS_DIR / 'primary_models_all.csv'],
    'aims_secondary_all': [STATS_DIR / 'aims_secondary_models_all.csv'],
    'aims_sensitivity_all': [STATS_DIR / 'aims_sensitivity_models_all.csv', STATS_DIR / 'sensitivity_models_all.csv'],
    'manuscript_primary': [STATS_DIR / 'manuscript_primary_results.csv'],
    'manuscript_primary_md': [STATS_DIR / 'manuscript_primary_results.md'],
    'qc_dashboard_md': [STATS_DIR / 'mvpa_l2_qc_dashboard.md'],
    'qc_subject_counts': [STATS_DIR / 'qc_subject_counts.csv'],
    'qc_missingness': [STATS_DIR / 'qc_missingness.csv'],
    'qc_model_status_counts': [STATS_DIR / 'qc_model_status_counts.csv'],
    'notebook_artifact_manifest': [STATS_DIR / 'notebook_artifact_manifest.csv'],
    'summary_md': [STATS_DIR / 'mvpa_l2_results_summary.md'],
}

REQUIRED_ARTIFACTS = [
    'subject_metrics',
    'aim1_primary',
    'aim2_primary',
    'aim3_primary',
    'aim4_primary',
    'aim5_primary',
    'aims_primary_all',
    'manuscript_primary',
]
OPTIONAL_ARTIFACTS = [key for key in ARTIFACT_CANDIDATES if key not in REQUIRED_ARTIFACTS]


def resolve_artifact_path(key):
    candidates = ARTIFACT_CANDIDATES[key]
    for path in candidates:
        if path.exists():
            return path
    return candidates[0]

ARTIFACTS = {key: resolve_artifact_path(key) for key in ARTIFACT_CANDIDATES}


def read_artifact(key, required=False):
    path = ARTIFACTS[key]
    if not path.exists():
        if required:
            checked = '; '.join(str(candidate) for candidate in ARTIFACT_CANDIDATES[key])
            raise FileNotFoundError(f'Missing required Hyak/post-Hyak artifact for {key}. Checked: {checked}')
        return pd.DataFrame()
    if path.suffix.lower() == '.md':
        return path.read_text(encoding='utf-8')
    return pd.read_csv(path)


def compact_columns(df):
    preferred = [
        'aim', 'scientific_question', 'analysis', 'sensitivity', 'feature_space', 'FeatureSpace', 'session',
        'Group', 'Drug', 'metric', 'metric_role', 'clinical_score', 'clinical_score_label', 'scr_index',
        'effect_label', 'term', 'estimate', 'ci_low', 'ci_high', 'p', 'p_value', 'q', 'q_within_question',
        'n', 'n_sad_subjects', 'n_hc_subjects', 'r2', 'status', 'formula',
    ]
    cols = [col for col in preferred if col in df.columns]
    return df[cols].copy() if cols else df.copy()


def show_table(title, key, required=False, n=40):
    display(Markdown(f'**{title}**'))
    obj = read_artifact(key, required=required)
    path = ARTIFACTS[key]
    if isinstance(obj, str):
        display(Markdown(obj))
        return obj
    if obj.empty:
        print(f'Optional artifact missing or empty: {path}')
        return obj
    view = compact_columns(obj).head(n)
    display(view)
    print(f'Loaded {len(obj):,} rows from {path}')
    return obj


def show_markdown(title, key, required=False):
    display(Markdown(f'**{title}**'))
    text = read_artifact(key, required=required)
    path = ARTIFACTS[key]
    if isinstance(text, str) and text.strip():
        display(Markdown(text))
        print(f'Loaded {path}')
    else:
        print(f'Optional artifact missing or empty: {path}')
    return text

metrics_df = read_artifact('subject_metrics')
artifact_status_df = pd.DataFrame([
    {
        'artifact': key,
        'required': key in REQUIRED_ARTIFACTS,
        'path': str(ARTIFACTS[key]),
        'exists': ARTIFACTS[key].exists(),
        'candidate_count': len(candidates),
        'rows': (len(pd.read_csv(ARTIFACTS[key])) if ARTIFACTS[key].exists() and ARTIFACTS[key].suffix.lower() == '.csv' else np.nan),
    }
    for key, candidates in ARTIFACT_CANDIDATES.items()
])


## Data Availability

This section reports exactly which Hyak/post-Hyak artifacts the notebook can see. Missing required artifacts are a workflow issue, not something the notebook should repair by computing replacement statistics.


In [ ]:
display(artifact_status_df.sort_values(['required', 'artifact'], ascending=[False, True]))
missing_required = artifact_status_df[artifact_status_df['required'] & ~artifact_status_df['exists']]
if not missing_required.empty:
    display(missing_required[['artifact', 'path']])
    raise FileNotFoundError('Required Hyak/post-Hyak statistics artifacts are missing. Run code/scripts/run_mvpa_l2_posthyak.sh or set MVPA_L2_ROOT to a complete output directory.')

if not metrics_df.empty:
    group_cols = [col for col in ['FeatureSpace', 'Group', 'Drug'] if col in metrics_df.columns]
    if group_cols:
        display(metrics_df.groupby(group_cols, dropna=False).size().rename('n_rows').reset_index())
    feature_spaces = sorted(metrics_df['FeatureSpace'].dropna().astype(str).unique()) if 'FeatureSpace' in metrics_df.columns else []
    print(f'Available feature spaces: {feature_spaces}')


## Pipeline Compliance Audit

The audit is read from exported Hyak/post-Hyak files. The notebook does not inspect checkpoints or source files to infer compliance.


In [ ]:
show_table('Notebook artifact manifest', 'notebook_artifact_manifest', n=120)
show_table('QC subject counts', 'qc_subject_counts', n=80)
show_table('QC missingness', 'qc_missingness', n=80)
show_table('QC model status counts', 'qc_model_status_counts', n=80)


## Aim 1: Group-Specific Neural Representation Identification

Aim 1 tables are loaded from exported decoding and sensitivity outputs.


In [ ]:
aim1_primary_df = show_table('Aim 1 primary decoding', 'aim1_primary', required=True, n=80)
show_table('Aim 1 feature-space sensitivity', 'aim1_feature_sensitivity', n=120)
show_table('Aim 1 SCR-cohort sensitivity', 'aim1_scr_sensitivity', n=120)
show_table('Aim 1 functional-drop tests: feature space', 'aim1_feature_functional_drop_tests', n=120)
show_table('Aim 1 functional-drop tests: SCR cohorts', 'aim1_scr_functional_drop_tests', n=120)


## Aim 1 Secondary: Haufe-Transformed Spatial Distribution

Haufe/SCR sensitivity summaries are loaded from post-Hyak exports when available.


In [ ]:
show_table('Aim 1/Aim 2 Haufe-SCR sensitivity summary', 'aim2_haufe_scr_sensitivity', n=80)
show_table('Aim 1/Aim 2 Haufe-SCR ROI distribution', 'aim2_haufe_scr_sensitivity_roi', n=120)


## Aim 2: SAD-HC Neural-Profile Difference Under Placebo

Aim 2 primary, secondary, and sensitivity statistics are loaded from post-Hyak model tables. Geometry and trajectory panel data are displayed as exported figure inputs only.


In [ ]:
aim2_primary_df = show_table('Aim 2 primary group difference', 'aim2_primary', required=True, n=80)
show_table('Aim 2 geometry panel input', 'aim2_geometry_panel', n=80)
show_table('Aim 2 trajectory panel input', 'aim2_trajectory_panel', n=80)
show_table('Aim 2 secondary subject metrics', 'aim2_secondary_subject_metrics', n=80)
show_table('Aim 2 secondary group difference', 'aim2_secondary', n=120)
show_table('Aim 2 sensitivity group difference', 'aim2_sensitivity', n=160)
show_table('Neural metric registry', 'neural_metric_registry', n=120)


## Aim 3: Clinical Relevance Of Neural Profiles Under Placebo

Aim 3 statistics are loaded from the primary, secondary, and sensitivity post-Hyak outputs.


In [ ]:
aim3_primary_df = show_table('Aim 3 primary clinical relevance', 'aim3_primary', required=True, n=120)
show_table('Aim 3 secondary clinical relevance', 'aim3_secondary', n=160)
show_table('Aim 3 sensitivity clinical relevance', 'aim3_sensitivity', n=160)


## Aim 4: Physiological Relevance Of Neural Profiles Under Placebo

Aim 4 neural-SCR convergence results are loaded from post-Hyak outputs, including the exported convergence matrix.


In [ ]:
aim4_primary_df = show_table('Aim 4 primary SCR convergence', 'aim4_primary', required=True, n=120)
show_table('Aim 4 secondary SCR convergence', 'aim4_secondary', n=160)
show_table('Aim 4 sensitivity SCR convergence', 'aim4_sensitivity', n=160)
show_table('Aim 4 convergence matrix', 'aim4_convergence_matrix', n=120)
show_table('Aim 4 convergence matrix wide', 'aim4_convergence_matrix_wide', n=120)


## Aim 5: Oxytocin Modulation Of Neural Profiles

Aim 5 interaction and sensitivity statistics are loaded from post-Hyak model outputs.


In [ ]:
aim5_primary_df = show_table('Aim 5 primary oxytocin modulation', 'aim5_primary', required=True, n=120)
show_table('Aim 5 secondary oxytocin modulation', 'aim5_secondary', n=160)
show_table('Aim 5 sensitivity oxytocin modulation', 'aim5_sensitivity', n=160)


## Integrated Manuscript Result Tables

Combined result tables and manuscript summaries are displayed directly from the post-Hyak exporter.


In [ ]:
show_table('All primary model rows', 'aims_primary_all', required=True, n=200)
show_table('All secondary model rows', 'aims_secondary_all', n=200)
show_table('All sensitivity model rows', 'aims_sensitivity_all', n=200)
show_table('Manuscript primary results', 'manuscript_primary', required=True, n=200)


## Saved Manuscript Artifacts And QC Dashboard

Markdown artifacts are rendered directly from the exported files.


In [ ]:
show_markdown('Manuscript primary results', 'manuscript_primary_md')
show_markdown('Aim 4 convergence matrix', 'aim4_convergence_md')
show_markdown('Reproducibility/QC dashboard', 'qc_dashboard_md')
show_markdown('Compact model summary', 'summary_md')


## Notebook Execution QA

This final gate verifies that required files are present and that the notebook source does not contain in-notebook statistical derivation calls.


In [ ]:
missing_required = artifact_status_df[artifact_status_df['required'] & ~artifact_status_df['exists']]
if not missing_required.empty:
    raise FileNotFoundError('Missing required artifacts: ' + ', '.join(missing_required['artifact'].astype(str)))

notebook_path = PROJECT_ROOT / 'code' / 'mvpa_l2.ipynb'
notebook_text = notebook_path.read_text(encoding='utf-8')
forbidden_tokens = [
    'stats' + 'models',
    'smf' + '.ols',
    'sm' + '.OLS',
    'ttest' + '_ind',
    'ttest' + '_1samp',
    'multiple' + 'tests',
    'permutation' + '_test_score',
    'pearson' + 'r(',
    'spearman' + 'r(',
    '.to' + '_csv(',
    '.to' + '_excel(',
    'joblib' + '.load',
]
violations = sorted(token for token in forbidden_tokens if token in notebook_text)
if violations:
    raise AssertionError('Notebook contains forbidden in-notebook statistics/export code: ' + ', '.join(violations))
print('Notebook QA passed: required Hyak/post-Hyak artifacts are present and no forbidden in-notebook statistical derivation calls were found.')


## Reporting Checklist

Before using these outputs in a manuscript, confirm that `MVPA_L2_ROOT` points to the intended mask-mode run, optional sensitivity outputs are labeled as missing when absent, and the rendered tables match the post-Hyak files listed in `notebook_artifact_manifest.csv`.
